**Recruitment Analytics**
**DBS Foundation Data Science Capstone 2026**

Notebook ini melakukan analisis eksplorasi terhadap tiga dataset rekrutmen:
- **Dataset_CV** — data profil kandidat
- **Dataset_Job** — data lowongan kerja internal
- **Glints_Job** — data lowongan kerja dari platform Glints
---

## 1. Setup & Import
Mengimpor semua library yang dibutuhkan untuk analisis data dan visualisasi interaktif.

In [36]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_dark'

print('Library berhasil diimpor.')

Library berhasil diimpor.


---
## 2. Muat Dataset

Memuat semua file dari direktori data (`DATA_DIR`) secara otomatis.

Fungsi `load_all_files()` mendukung format: `.csv`, `.xlsx`, `.xls`, `.parquet`, `.json`.

In [37]:
DATA_DIR = r"D:\DBS_Foundation-2026\data_clean_DS"

SUPPORTED_EXT = {'.csv', '.xlsx', '.xls', '.parquet', '.json'}

def load_all_files(data_dir):
    dataframes = {}
    for fname in sorted(os.listdir(data_dir)):
        name, ext = os.path.splitext(fname)
        if ext.lower() not in SUPPORTED_EXT:
            continue
        fpath = os.path.join(data_dir, fname)
        if ext.lower() == '.csv':
            df = pd.read_csv(fpath)
        elif ext.lower() in ('.xlsx', '.xls'):
            df = pd.read_excel(fpath)
        elif ext.lower() == '.parquet':
            df = pd.read_parquet(fpath)
        elif ext.lower() == '.json':
            df = pd.read_json(fpath)
        dataframes[name] = df
        print(f'{name:<35} {df.shape[0]:>7,} baris  x  {df.shape[1]} kolom')
    print(f'\nTotal file dimuat: {len(dataframes)}')
    return dataframes

datasets = load_all_files(DATA_DIR)

Dataset_CV                           11,915 baris  x  4 kolom
Dataset_Glints_Job                      565 baris  x  14 kolom
Dataset_Job                          11,027 baris  x  9 kolom

Total file dimuat: 3


**Menetapkan referensi variabel untuk ketiga dataframe utama yang akan digunakan sepanjang analisis.**

In [38]:
# Sesuaikan nama key dengan nama file di folder
df_cv     = datasets['Dataset_CV']
df_job    = datasets['Dataset_Job']
df_glints = datasets['Dataset_Glints_Job']

print('df_cv     :', df_cv.shape)
print('df_job    :', df_job.shape)
print('df_glints :', df_glints.shape)

df_cv     : (11915, 4)
df_job    : (11027, 9)
df_glints : (565, 14)


**Mendefinisikan konstanta nama kolom untuk setiap dataset.**

In [39]:
# Sesuaikan nilai string di bawah dengan nama kolom aktual di masing-masing file

# Dataset_CV
CV_EDUCATION_COL  = 'pendidikan'
CV_EXPERIENCE_COL = 'pengalaman_tahun'
CV_SKILL_COL      = 'skill'

# Dataset_Job
JOB_POSITION_COL  = 'posisi'
JOB_MIN_EXP_COL   = 'min_pengalaman'
JOB_EDU_COL       = 'pendidikan_min'
JOB_GENDER_COL    = 'gender'
JOB_AGE_MIN_COL   = 'usia_min'

# Glints_Job
GLINTS_ROLE_COL     = 'kategori_peran'
GLINTS_CITY_COL     = 'kota'
GLINTS_WORK_SYS_COL = 'sistem_kerja'
GLINTS_WORK_TIME_COL= 'tipe_waktu'
GLINTS_SALARY_MIN   = 'gaji_min'
GLINTS_SALARY_MAX   = 'gaji_max'
GLINTS_SKILL_COL    = 'skill'

print('Konstanta kolom didefinisikan.')

# Validasi kolom ada di dataframe
missing_cols = []
for df, col, label in [
    (df_cv,     CV_EDUCATION_COL,   'CV_EDUCATION_COL'),
    (df_cv,     CV_EXPERIENCE_COL,  'CV_EXPERIENCE_COL'),
    (df_cv,     CV_SKILL_COL,       'CV_SKILL_COL'),
    (df_job,    JOB_POSITION_COL,   'JOB_POSITION_COL'),
    (df_job,    JOB_MIN_EXP_COL,    'JOB_MIN_EXP_COL'),
    (df_job,    JOB_EDU_COL,        'JOB_EDU_COL'),
    (df_job,    JOB_GENDER_COL,     'JOB_GENDER_COL'),
    (df_glints, GLINTS_ROLE_COL,    'GLINTS_ROLE_COL'),
    (df_glints, GLINTS_CITY_COL,    'GLINTS_CITY_COL'),
    (df_glints, GLINTS_WORK_SYS_COL,'GLINTS_WORK_SYS_COL'),
    (df_glints, GLINTS_WORK_TIME_COL,'GLINTS_WORK_TIME_COL'),
    (df_glints, GLINTS_SALARY_MIN,  'GLINTS_SALARY_MIN'),
    (df_glints, GLINTS_SALARY_MAX,  'GLINTS_SALARY_MAX'),
    (df_glints, GLINTS_SKILL_COL,   'GLINTS_SKILL_COL'),
]:
    if col not in df.columns:
        missing_cols.append(f'{label} -> "{col}" tidak ditemukan')

if missing_cols:
    print('\nKolom tidak ditemukan — sesuaikan nilai konstanta di atas:')
    for m in missing_cols:
        print(f'  {m}')
else:
    print('Semua kolom valid.')

Konstanta kolom didefinisikan.

Kolom tidak ditemukan — sesuaikan nilai konstanta di atas:
  CV_EXPERIENCE_COL -> "pengalaman_tahun" tidak ditemukan
  JOB_MIN_EXP_COL -> "min_pengalaman" tidak ditemukan
  JOB_EDU_COL -> "pendidikan_min" tidak ditemukan
  GLINTS_CITY_COL -> "kota" tidak ditemukan
  GLINTS_WORK_TIME_COL -> "tipe_waktu" tidak ditemukan
  GLINTS_SALARY_MIN -> "gaji_min" tidak ditemukan
  GLINTS_SALARY_MAX -> "gaji_max" tidak ditemukan


**Mendefinisikan dua fungsi helper yang akan digunakan berulang di seluruh notebook:**
- `split_skills()` — memecah kolom skill yang berupa string CSV menjadi baris individual
- `top_n_cities()` — mengambil N kota dengan lowongan terbanyak

In [40]:
def split_skills(series, sep=','):
    return series.dropna().str.split(sep).explode().str.strip().replace('', pd.NA).dropna()

def top_n_cities(df, col, n=5):
    return df[col].value_counts().head(n).index.tolist()

---
## 2. EDA (Exploratory Data Analysis) 

**Bagian ini memeriksa struktur dan kualitas data secara menyeluruh sebelum menjawab business questions.**

Pemeriksaan meliputi:
1. Preview sampel data
2. Tipe data tiap kolom
3. Missing values
4. Statistik deskriptif numerik
5. Distribusi kolom kategorik
6. Visualisasi distribusi numerik

### 2.1 Preview Dataset

**Menampilkan 5 baris pertama dari masing-masing dataset beserta daftar nama kolomnya**

In [41]:
print('=== Preview Dataset ===')

for name, df in [('Dataset_CV', df_cv), ('Dataset_Job', df_job), ('Glints_Job', df_glints)]:
    print(f'\n{name}:')
    display(df.head())
    print('Kolom:', df.columns.tolist())

=== Preview Dataset ===

Dataset_CV:


,pendidikan,pengalaman,skill,detail
0,Magister (S2),5-10 tahun,"AWS, AWS Glue, Amazon Redshift, Amazon S3, Apa...","menguasai Python, Java, Scala, SQL, Hadoop, Ap..."
1,tidak ada pendidikan spesifik,10+ tahun,"Assembly, Business Analysis, Business Requirem...",Menguasai Strategic Planning e Business-System...
2,Magister (S2),2-3 tahun,"API, AWS, Agile, Agile Methodology, Angular, B...","Menguasai JavaScript, HTML/CSS, dan Python ser..."
3,Magister (S2),3-5 tahun,"AWS, Agile, Agile Methodology, Angular, Asana,...","menguasai JavaScript, Python, Java, React, Ang..."
4,Magister (S2),5-10 tahun,"AWS, AWS Lambda, Agile, Agile Methodology, Ama...","Menguasai berbagai platform cloud, termasuk AW..."


Kolom: ['pendidikan', 'pengalaman', 'skill', 'detail']

Dataset_Job:


,posisi,pendidikan,pengalaman,gender,usia,skill,kualifikasi,kualifikasi_asli,skill_original
0,Software Engineer,terbuka untuk semua jenjang dan jurusan,0-1 tahun,tanpa ketentuan,tanpa batasan usia,"Java, Python","Menguasai Java dan Python, mampu mengembangkan...",Develop software,"Java, Python"
1,Data Analyst,terbuka untuk semua jenjang dan jurusan,1-3 tahun,tanpa ketentuan,tanpa batasan usia,"SQL, Microsoft Excel","Menguasai SQL dan Excel, berpengalaman 1-3 tah...",Analyze data,"SQL, Excel"
2,Network Engineer,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,"Cisco, WAN","Menguasai teknologi Cisco dan WAN, berpengalam...",Maintain networks,"Cisco, WAN"
3,Cloud Architect,terbuka untuk semua jenjang dan jurusan,5+ tahun,tanpa ketentuan,tanpa batasan usia,"AWS, Microsoft Azure",Menguasai teknologi cloud seperti AWS dan Azur...,Design cloud,"AWS, Azure"
4,Cybersecurity Analyst,terbuka untuk semua jenjang dan jurusan,3-5 tahun,tanpa ketentuan,tanpa batasan usia,Cybersecurity,"Menguasai Cybersecurity, berpengalaman 3-5 tah...",Protect data,Cybersecurity


Kolom: ['posisi', 'pendidikan', 'pengalaman', 'gender', 'usia', 'skill', 'kualifikasi', 'kualifikasi_asli', 'skill_original']

Glints_Job:


,tautan,posisi,perusahaan,lokasi,gaji,pendidikan,pengalaman,gender,usia,skill,kualifikasi,tipe_waktu_kerja,sistem_kerja,kategori_peran
0,https://glints.com/id/opportunities/jobs/ai-ml...,AI / ML Engineer (Model Development Support),eVantage HR,"Jakarta Selatan, DKI Jakarta",Rp20.000.000 - 25.000.000/Bulan,Minimal Sarjana (S1),5 - 10 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Python, Evaluation Metrix, Machine Learning, A...","Role Summary\nSupport AI model development, tr...",Penuh Waktu,Kerja di kantor,AI Engineer
1,https://glints.com/id/opportunities/jobs/ai-cr...,AI Creative Engineer,Willscale,"Jakarta Selatan, DKI Jakarta",Rp4.500.000 - 6.000.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Creative Thinking, Artificial Intelligence, Vi...",Are you a tech-driven innovator who loves to p...,Penuh Waktu,Remote/Dari rumah,Posisi Kecerdasan Buatan Lainnya
2,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,PT DIGITAL SEJAHTERA NUSANTARA,"Jakarta Pusat, DKI Jakarta",Rp7.000.000 - 11.000.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,tanpa batasan usia,"Pattern Recognition, Machine Learning, Artific...",Requirements:\nHands-on experience implementin...,Kontrak,Kerja di kantor,AI Engineer
3,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,PT Kotagede Jewellery Group,"Kab. Sleman, DI Yogyakarta",Rp4.000.000 - 5.500.000/Bulan,Minimal Sarjana (S1),1 - 3 tahun pengalaman,tanpa ketentuan,25-35 tahun,"Artificial Intelligence, Python, Data Mining, ...",Kami adalah perusahaan dengan tim yang diisi o...,Penuh Waktu,Kerja di kantor,AI Engineer
4,https://glints.com/id/opportunities/jobs/ai-en...,AI Engineer,Zando Agency,"Jakarta Utara, DKI Jakarta",Rp13.000.000 - 17.000.000/Bulan,Minimal Sarjana (S1),3 - 5 tahun pengalaman,tanpa ketentuan,20-30 tahun,"Machine Learning, Cloud Platform System, Pytho...",Kamu siap bangun masa depan bareng tim yang ge...,Penuh Waktu,Kerja di kantor,AI Engineer


Kolom: ['tautan', 'posisi', 'perusahaan', 'lokasi', 'gaji', 'pendidikan', 'pengalaman', 'gender', 'usia', 'skill', 'kualifikasi', 'tipe_waktu_kerja', 'sistem_kerja', 'kategori_peran']


### 2.2 Tipe Data

- Mengecek tipe data (`dtype`) setiap kolom. Kolom yang seharusnya numerik tetapi bertipe `object`
- perlu dikonversi secara eksplisit sebelum analisis.

In [42]:
print('=== Tipe Data ===')

for name, df in [('Dataset_CV', df_cv), ('Dataset_Job', df_job), ('Glints_Job', df_glints)]:
    print(f'\n{name}:')
    display(df.dtypes.to_frame(name='dtype'))

=== Tipe Data ===

Dataset_CV:


,dtype
pendidikan,object
pengalaman,object
skill,object
detail,object



Dataset_Job:


,dtype
posisi,object
pendidikan,object
pengalaman,object
gender,object
usia,object
skill,object
kualifikasi,object
kualifikasi_asli,object
skill_original,object



Glints_Job:


,dtype
tautan,object
posisi,object
perusahaan,object
lokasi,object
gaji,object
pendidikan,object
pengalaman,object
gender,object
usia,object
skill,object


### 2.3 Missing Values

**Menghitung jumlah dan persentase nilai kosong per kolom.**

Hanya kolom dengan missing value yang ditampilkan.

In [43]:
print('=== Missing Values ===')

for name, df in [('Dataset_CV', df_cv), ('Dataset_Job', df_job), ('Glints_Job', df_glints)]:
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    report  = pd.DataFrame({'Missing': missing, 'Persen (%)': pct})
    report  = report[report['Missing'] > 0]
    print(f'\n{name}:')
    if report.empty:
        print('  Tidak ada missing values.')
    else:
        display(report)

=== Missing Values ===

Dataset_CV:


,Missing,Persen (%)
detail,857,7.19



Dataset_Job:


,Missing,Persen (%)
skill,2,0.02
kualifikasi,3,0.03
kualifikasi_asli,352,3.19



Glints_Job:
  Tidak ada missing values.


**Hasil laporan missing values selesai.**
- Kolom dengan missing > 50% perlu dipertimbangkan untuk di-drop atau diimputasi.
- Kolom skill yang sering kosong wajar terjadi pada dataset lowongan.

### 2.4 Statistik Deskriptif

**Menghitung statistik ringkasan (mean, std, min, max, kuartil) untuk semua kolom numerik.**

In [44]:
print('=== Statistik Deskriptif ===')

for name, df in [('Dataset_CV', df_cv), ('Dataset_Job', df_job), ('Glints_Job', df_glints)]:
    num_cols = df.select_dtypes(include='number')
    print(f'\n{name}:')
    if num_cols.empty:
        print('  Tidak ada kolom numerik.')
    else:
        display(num_cols.describe().T)

=== Statistik Deskriptif ===

Dataset_CV:
  Tidak ada kolom numerik.

Dataset_Job:
  Tidak ada kolom numerik.

Glints_Job:
  Tidak ada kolom numerik.


Pada hasil statistik deskriptif tidak terdapat kolom numerik sehingga tidak ada hasil yang ditampilkan 

### 2.5 Distribusi Kolom Kategorik

**Menampilkan frekuensi nilai unik (top 10) untuk kolom kategorik utama di setiap dataset.**

In [45]:
print('=== Distribusi Kolom Kategorik ===')

for name, df, cols in [
    ('Dataset_CV',  df_cv,     [CV_EDUCATION_COL]),
    ('Dataset_Job', df_job,    [JOB_EDU_COL, JOB_GENDER_COL]),
    ('Glints_Job',  df_glints, [GLINTS_ROLE_COL, GLINTS_CITY_COL, GLINTS_WORK_SYS_COL, GLINTS_WORK_TIME_COL]),
]:
    print(f'\n{name}:')
    for col in cols:
        if col in df.columns:
            vc  = df[col].value_counts()
            pct = (vc / len(df) * 100).round(2)
            tbl = pd.DataFrame({'Jumlah': vc, 'Persen (%)': pct})
            print(f'  [{col}]')
            display(tbl.head(10))

=== Distribusi Kolom Kategorik ===

Dataset_CV:
  [pendidikan]


,Jumlah,Persen (%)
pendidikan,,
Magister (S2),9454,79.35
tidak ada pendidikan spesifik,1056,8.86
Sarjana (S1),785,6.59
PhD (S3),449,3.77
Diploma 3 (D3),62,0.52
Diploma 1/2 (D1/D2),55,0.46
PhD (S3)),28,0.23
SMA/SMK,26,0.22



Dataset_Job:
  [gender]


,Jumlah,Persen (%)
gender,,
tanpa ketentuan,10937,99.18
Laki-laki saja,80,0.73
Perempuan saja,10,0.09



Glints_Job:
  [kategori_peran]


,Jumlah,Persen (%)
kategori_peran,,
Full Stack Developer,64,11.33
IT Support,63,11.15
Backend Developer,51,9.03
AI Engineer,36,6.37
Frontend Developer,35,6.19
Network Engineer,32,5.66
Data Analyst,32,5.66
DevOps Engineer,31,5.49
Technical Support Engineer,28,4.96


  [sistem_kerja]


,Jumlah,Persen (%)
sistem_kerja,,
Kerja di kantor,482,85.31
Hybrid,56,9.91
Remote/Dari rumah,27,4.78


Nilai yang tidak konsisten (typo, kapitalisasi berbeda) perlu di-clean sebelum dianalisis lebih lanjut.

### 2.6 Distribusi Kolom Numerik

**Memvisualisasikan distribusi kolom numerik kunci: pengalaman kandidat, syarat pengalaman lowongan, serta rentang gaji minimum dan maksimum dari Glints.**

In [46]:
import re as _re

COLOR_SEQ       = __import__('plotly.express', fromlist=['colors']).colors.qualitative.Bold
TEMPLATE        = 'plotly_dark'

# --- Parse helper: "Rp20.000.000 - 25.000.000/Bulan" -> median float ---
def _parse_gaji(series):
    clean = (
        series.astype(str)
        .str.replace(r'Rp', '', regex=False)
        .str.replace(r'/[Bb]ulan', '', regex=True)
        .str.replace(r'\.', '', regex=True)
        .str.strip()
    )
    split = clean.str.split(r'\s*-\s*', expand=True, regex=True)
    lo = pd.to_numeric(split[0].str.strip(), errors='coerce')
    hi = pd.to_numeric(split[1].str.strip(), errors='coerce') if split.shape[1] > 1 else lo
    return ((lo + hi) / 2).dropna()

# --- Parse helper: "2-3 tahun", "5+ tahun", "< 1 tahun" -> float ---
def _parse_exp(series):
    def _cv(v):
        s = str(v).strip().lower()
        if _re.match(r'^<\s*\d', s): return 0.0
        m = _re.match(r'^(\d+)\+', s);
        if m: return float(m.group(1))
        m = _re.match(r'^(\d+)\s*[-\u2013]\s*(\d+)', s)
        if m: return float(m.group(1))
        m = _re.match(r'^(\d+)', s)
        if m: return float(m.group(1))
        return float('nan')
    return series.apply(_cv).dropna()

numeric_targets = [
    (_parse_exp(df_cv['pengalaman']),           'Pengalaman (tahun) — Dataset_CV',        'Tahun Pengalaman',     '#2196F3'),
    (_parse_exp(df_job['pengalaman']),           'Pengalaman Minimum (tahun) — Dataset_Job','Tahun Pengalaman',    '#4CAF50'),
    (_parse_gaji(df_glints['gaji']),             'Gaji Median (Rp) — Glints_Job',          'Gaji Median (Rp)',     '#FF9800'),
]

for series, title, xlabel, color in numeric_targets:
    if series.empty:
        print(f'[SKIP] {title} — tidak ada data numerik.')
        continue
    fig = px.histogram(
        series,
        nbins=30,
        title=title,
        labels={'value': xlabel, 'count': 'Frekuensi'},
        template=TEMPLATE,
        color_discrete_sequence=[color]
    )
    fig.update_layout(showlegend=False, height=400,
                      xaxis_title=xlabel, yaxis_title='Frekuensi')
    fig.show()


Distribusi miring ke kanan (right-skewed) pada gaji mengindikasikan adanya posisi bergaji sangat tinggi yang menarik median ke atas.

---
## 3. Business Questions

**Menjawab 10 pertanyaan bisnis utama menggunakan data yang telah dimuat.**

Setiap BQ disertai tabel ringkasan dan visualisasi interaktif.

**Helper: `print_section()`**

Fungsi utilitas untuk mencetak pemisah, judul, tabel, dan jawaban akhir setiap business question
secara konsisten.

In [47]:
def print_section(title, df=None, answer=None):
    sep = '-' * 60
    print(sep)
    print(f'  {title}')
    print(sep)
    if df is not None:
        display(df)
    if answer is not None:
        print()
        print(f'  JAWABAN : {answer}')
    print()

---
### BQ 1 - Distribusi Tingkat Pendidikan Kandidat

**Pertanyaan :** Bagaimana distribusi tingkat pendidikan kandidat, dan apakah S1 mendominasi lebih dari 50%?

**Chart :** Pie Chart + Bar Chart Horizontal

Menghitung proporsi setiap jenjang pendidikan dari `Dataset_CV`, lalu mengecek apakah
kandidat berlatar belakang S1/Sarjana/Bachelor melebihi 50% dari total.

In [48]:
edu_counts = df_cv[CV_EDUCATION_COL].value_counts().reset_index()
edu_counts.columns = ['Pendidikan', 'Jumlah']
edu_counts['Persen (%)'] = (edu_counts['Jumlah'] / edu_counts['Jumlah'].sum() * 100).round(2)

s1_rows = edu_counts[edu_counts['Pendidikan'].str.contains('S1|Sarjana|Bachelor', case=False, na=False)]
s1_pct  = s1_rows['Persen (%)'].sum()
answer  = f'S1 mendominasi >50%? {"YA" if s1_pct > 50 else "TIDAK"} ({s1_pct:.2f}%)'

print_section('BQ 1 — Distribusi Pendidikan Kandidat', edu_counts, answer)

------------------------------------------------------------
  BQ 1 — Distribusi Pendidikan Kandidat
------------------------------------------------------------


,Pendidikan,Jumlah,Persen (%)
0,Magister (S2),9454,79.35
1,tidak ada pendidikan spesifik,1056,8.86
2,Sarjana (S1),785,6.59
3,PhD (S3),449,3.77
4,Diploma 3 (D3),62,0.52
5,Diploma 1/2 (D1/D2),55,0.46
6,PhD (S3)),28,0.23
7,SMA/SMK,26,0.22



  JAWABAN : S1 mendominasi >50%? TIDAK (6.59%)



**Visualisasi BQ 1:** Pie chart menampilkan proporsi tiap jenjang dengan bar chart horizontal yang memudahkan perbandingan jumlah absolut antar kategori.

In [49]:
# Visualisasi BQ 1
edu_fig = px.pie(edu_counts, names='Pendidikan', values='Jumlah',
                 title='BQ 1 — Distribusi Pendidikan Kandidat')
edu_fig.show()

edu_bar = px.bar(edu_counts.sort_values('Jumlah'),
                 x='Jumlah', y='Pendidikan', orientation='h',
                 text='Persen (%)', title='BQ 1 - Distribusi Pendidikan (Bar)')
edu_bar.show()

---
### BQ 2 - Distribusi Level Pengalaman Lowongan

**Pertanyaan :** Berapa persen lowongan yang menargetkan entry-level (0–2 th), mid-level (2–5 th), dan senior (5+ th)?

**Chart :** Donut Chart

Kolom pengalaman minimum dari `Dataset_Job` diekstrak menggunakan regex (menangani format
teks seperti '1-3 tahun', '5+ tahun'), kemudian dikelompokkan ke dalam tiga bucket level.

In [50]:
if 'experience_level' not in df_job.columns:
    if 'experience_level' not in df_job.columns:
        # gunakan kolom min_pengalaman jika ada, fallback ke 'pengalaman' jika tersedia
        source_col = JOB_MIN_EXP_COL if JOB_MIN_EXP_COL in df_job.columns else ('pengalaman' if 'pengalaman' in df_job.columns else None)
        if source_col is None:
            # tidak ada kolom sumber, buat kolom numeric kosong supaya pd.cut tidak error
            df_job[JOB_MIN_EXP_COL] = np.nan
        else:
            # ekstrak angka pertama dari teks (mis. "1-3 tahun", "5+ tahun", "3 - 5 tahun", "1 tahun")
            df_job[JOB_MIN_EXP_COL] = (
                df_job[source_col].astype(str).str.lower().str.extract(r'(\d+(?:\.\d+)?)')[0]
            )
            df_job[JOB_MIN_EXP_COL] = pd.to_numeric(df_job[JOB_MIN_EXP_COL], errors='coerce')

        df_job['experience_level'] = pd.cut(
            df_job[JOB_MIN_EXP_COL],
            bins=[-1, 2, 5, 100],
            labels=['Entry-Level (0-2 th)', 'Mid-Level (2-5 th)', 'Senior (5+ th)']
        )

    level_counts = df_job['experience_level'].value_counts().reset_index()
    level_counts.columns = ['Level', 'Jumlah']
    level_counts['Persen (%)'] = (level_counts['Jumlah'] / level_counts['Jumlah'].sum() * 100).round(2)

    entry_pct = level_counts.loc[level_counts['Level'].astype(str).str.contains('Entry', case=False), 'Persen (%)'].sum()
    answer    = f'Entry-level mendominasi >40%? {"YA" if entry_pct > 40 else "TIDAK"} ({entry_pct:.2f}%)'

    print_section('BQ 2 — Level Pengalaman Lowongan', level_counts, answer)
    df_job['experience_level'] = pd.cut(
        df_job[JOB_MIN_EXP_COL],
        bins=[-1, 2, 5, 100],
        labels=['Entry-Level (0-2 th)', 'Mid-Level (2-5 th)', 'Senior (5+ th)']
    )

level_counts = df_job['experience_level'].value_counts().reset_index()
level_counts.columns = ['Level', 'Jumlah']
level_counts['Persen (%)'] = (level_counts['Jumlah'] / level_counts['Jumlah'].sum() * 100).round(2)

entry_pct = level_counts.loc[level_counts['Level'].astype(str).str.contains('Entry', case=False), 'Persen (%)'].sum()
answer    = f'Entry-level mendominasi >40%? {"YA" if entry_pct > 40 else "TIDAK"} ({entry_pct:.2f}%)'

print_section('BQ 2 — Level Pengalaman Lowongan', level_counts, answer)

------------------------------------------------------------
  BQ 2 — Level Pengalaman Lowongan
------------------------------------------------------------


,Level,Jumlah,Persen (%)
0,Entry-Level (0-2 th),1456,52.95
1,Mid-Level (2-5 th),999,36.33
2,Senior (5+ th),295,10.73



  JAWABAN : Entry-level mendominasi >40%? YA (52.95%)

------------------------------------------------------------
  BQ 2 — Level Pengalaman Lowongan
------------------------------------------------------------


,Level,Jumlah,Persen (%)
0,Entry-Level (0-2 th),1456,52.95
1,Mid-Level (2-5 th),999,36.33
2,Senior (5+ th),295,10.73



  JAWABAN : Entry-level mendominasi >40%? YA (52.95%)



**Visualisasi BQ 2 :** Donut chart menampilkan proporsi masing-masing level pengalaman yang diminta oleh perusahaan.

In [51]:
# Visualisasi BQ 2
fig2 = px.pie(level_counts, names='Level', values='Jumlah', hole=0.45,
              title='BQ 2 — Level Pengalaman Lowongan (Donut)')
fig2.show()

---
### BQ 3 - Top 5 Kota dengan Lowongan Terbanyak & Komposisi Sistem Kerja

**Pertanyaan :** Kota mana top 5 terbanyak dan bagaimana komposisi WFO/WFH/Hybrid di setiap kota?

**Chart :** Horizontal Stacked Bar Chart

Mengambil 5 kota dengan jumlah lowongan terbanyak dari `Glints_Job`, lalu membuat pivot table untuk melihat komposisi sistem kerja di masing-masing kota.

In [52]:
GLINTS_CITY_COL = 'lokasi'
top5 = df_glints[GLINTS_CITY_COL].value_counts().head(5).index.tolist()
df_top5 = df_glints[df_glints[GLINTS_CITY_COL].isin(top5)]

pivot          = df_top5.groupby([GLINTS_CITY_COL, GLINTS_WORK_SYS_COL]).size().unstack(fill_value=0)
pivot['Total'] = pivot.sum(axis=1)
pivot_pct      = pivot.drop(columns='Total').div(pivot['Total'], axis=0).mul(100).round(2)
pivot_pct.insert(0, 'Total Lowongan', pivot['Total'])

answer = f'Top 5 kota: {" > ".join(top5)}'

print_section(f'BQ 3 — Top 5 Kota & Sistem Kerja (total: {len(df_glints):,} lowongan)', pivot_pct, answer)

------------------------------------------------------------
  BQ 3 — Top 5 Kota & Sistem Kerja (total: 565 lowongan)
------------------------------------------------------------


sistem_kerja,Total Lowongan,Hybrid,Kerja di kantor,Remote/Dari rumah
lokasi,,,,
"Jakarta Barat, DKI Jakarta",48,14.58,79.17,6.25
"Jakarta Pusat, DKI Jakarta",61,9.84,88.52,1.64
"Jakarta Selatan, DKI Jakarta",123,14.63,80.49,4.88
"Jakarta Utara, DKI Jakarta",33,0.00,93.94,6.06
"Surabaya, Jawa Timur",34,20.59,73.53,5.88



  JAWABAN : Top 5 kota: Jakarta Selatan, DKI Jakarta > Jakarta Pusat, DKI Jakarta > Jakarta Barat, DKI Jakarta > Surabaya, Jawa Timur > Jakarta Utara, DKI Jakarta



**Visualisasi BQ 3 :** Stacked bar horizontal memperlihatkan volume lowongan sekaligus proporsi sistem kerja (WFO/WFH/Hybrid) per kota.

In [53]:
COLOR_WFO      = '#2196F3'
COLOR_WFH      = '#4CAF50'
COLOR_HYBRID   = '#FF9800'
TEMPLATE       = 'plotly_dark'
COLOR_SEQ      = __import__('plotly.express', fromlist=['colors']).colors.qualitative.Bold

_CITY_COL    = 'kota_singkat'
_WORKSYS_COL = 'sistem_kerja_label'

WORK_SYS_LABELS = {
    'Kerja di kantor'  : 'WFO',
    'Remote/Dari rumah': 'WFH',
    'Hybrid'           : 'Hybrid',
}

df_g3 = df_glints.copy()
df_g3[_CITY_COL]    = df_g3['lokasi'].str.split(',').str[0].str.strip()
df_g3[_WORKSYS_COL] = df_g3['sistem_kerja'].map(WORK_SYS_LABELS).fillna(df_g3['sistem_kerja'])

# Top 5 kota diurutkan descending (bar terpanjang di atas)
top5_ordered = df_g3[_CITY_COL].value_counts().head(5).index.tolist()

pivot = (
    df_g3[df_g3[_CITY_COL].isin(top5_ordered)]
    .groupby([_CITY_COL, _WORKSYS_COL])
    .size()
    .reset_index(name='Jumlah')
)

# Hitung total per kota untuk pengurutan
kota_total = pivot.groupby(_CITY_COL)['Jumlah'].sum().sort_values(ascending=True)
sorted_kota = kota_total.index.tolist()   # ascending untuk horizontal bar

color_map = {'WFO': COLOR_WFO, 'WFH': COLOR_WFH, 'Hybrid': COLOR_HYBRID}

fig3 = px.bar(
    pivot,
    x='Jumlah',
    y=_CITY_COL,
    color=_WORKSYS_COL,
    orientation='h',
    barmode='stack',
    text='Jumlah',
    color_discrete_map=color_map,
    category_orders={_CITY_COL: sorted_kota},
    title='BQ 3 — Top 5 Kota dengan Lowongan Terbanyak & Komposisi Sistem Kerja',
    labels={_CITY_COL: 'Kota', 'Jumlah': 'Jumlah Lowongan', _WORKSYS_COL: 'Sistem Kerja'}
)
fig3.update_layout(template=TEMPLATE, height=420, legend_title='Sistem Kerja')
fig3.update_traces(textposition='inside')
fig3.show()


---
### BQ 4 - Median Gaji per Kategori Peran

**Pertanyaan :** Dari kategori peran dengan minimal 5 lowongan, mana 10 dengan median gaji tertinggi dan terendah?

**Chart :** Horizontal Bar Chart (Top 10 & Bottom 10) + Box Plot

Kolom `salary_median` dibuat dari rata-rata `gaji_min` dan `gaji_max`. Hanya kategori dengan ≥ 5 lowongan yang diikutkan agar statistik representatif.

In [54]:
if 'salary_median' not in df_glints.columns:
    df_glints['gaji_min'] = df_glints['gaji'].str.extract(r'(\d+)')[0].astype(float) * 1_000_000
    df_glints['gaji_max'] = df_glints['gaji'].str.extract(r'(\d+)(?!.*\d)')[0].astype(float) * 1_000_000
    df_glints['salary_median'] = (df_glints['gaji_min'] + df_glints['gaji_max']) / 2

COLOR_WFH = '#00CC96'
COLOR_THRESHOLD = '#EF553B'
COLOR_SEQ = px.colors.qualitative.Plotly
TEMPLATE = 'plotly_dark'

df_salary = df_glints[df_glints['salary_median'].notna()].copy()
role_stats = (
    df_salary.groupby(GLINTS_ROLE_COL)
    .agg(
        median_gaji=('salary_median', 'median'),
        count=(GLINTS_ROLE_COL, 'count'),
        salary_range_min=(GLINTS_SALARY_MIN, 'min'),
        salary_range_max=(GLINTS_SALARY_MAX, 'max')
    )
    .reset_index()
    .query('count >= 5')
    .sort_values('median_gaji', ascending=False)
    .reset_index(drop=True)
)

if not role_stats.empty:
    top_med = role_stats.iloc[0]['median_gaji']
    bot_med = role_stats.iloc[-1]['median_gaji']
    answer = (
        f'Tertinggi: {role_stats.iloc[0][GLINTS_ROLE_COL]} (Rp {top_med/1e6:.2f} Jt) | '
        f'Terendah: {role_stats.iloc[-1][GLINTS_ROLE_COL]} (Rp {bot_med/1e6:.2f} Jt) | '
        f'Rentang: Rp {(top_med - bot_med)/1e6:.2f} Jt'
    )
else:
    answer = 'Data gaji tidak tersedia atau tidak ada kategori dengan setidaknya 5 lowongan.'

print_section('BQ 4 — Top 10 Median Gaji Tertinggi', role_stats.head(10)[[GLINTS_ROLE_COL, 'median_gaji', 'salary_range_min', 'salary_range_max', 'count']], answer=None)
print_section('BQ 4 — Bottom 10 Median Gaji Terendah', role_stats.tail(10)[[GLINTS_ROLE_COL, 'median_gaji', 'salary_range_min', 'salary_range_max', 'count']], answer)

------------------------------------------------------------
  BQ 4 — Top 10 Median Gaji Tertinggi
------------------------------------------------------------


,kategori_peran,median_gaji,salary_range_min,salary_range_max,count
0,Project Manager,4250000.0,4000000.0,703000000.0,20
1,Data Engineer,4000000.0,1000000.0,0.0,26
2,Product Manager,4000000.0,1000000.0,0.0,13
3,Cybersecurity,3500000.0,6000000.0,0.0,7
4,DevOps Engineer,3000000.0,2000000.0,0.0,21
5,Backend Developer,3000000.0,1000000.0,0.0,41
6,Mobile Developer,2750000.0,1000000.0,0.0,12
7,AI Engineer,2500000.0,1000000.0,0.0,30
8,QA Engineer,2500000.0,1000000.0,500000000.0,17
9,Data Analyst,2500000.0,1000000.0,0.0,25



------------------------------------------------------------
  BQ 4 — Bottom 10 Median Gaji Terendah
------------------------------------------------------------


,kategori_peran,median_gaji,salary_range_min,salary_range_max,count
7,AI Engineer,2500000.0,1000000.0,0.0,30
8,QA Engineer,2500000.0,1000000.0,500000000.0,17
9,Data Analyst,2500000.0,1000000.0,0.0,25
10,Database Administrator,2500000.0,1000000.0,1000000.0,13
11,Frontend Developer,2500000.0,1000000.0,1000000.0,27
12,Network Engineer,2500000.0,1000000.0,0.0,28
13,System Administrator,2000000.0,2000000.0,0.0,6
14,Full Stack Developer,2000000.0,1000000.0,0.0,53
15,IT Support,1500000.0,1000000.0,709000000.0,49
16,Technical Support Engineer,1500000.0,2000000.0,0.0,15



  JAWABAN : Tertinggi: Project Manager (Rp 4.25 Jt) | Terendah: Technical Support Engineer (Rp 1.50 Jt) | Rentang: Rp 2.75 Jt



**Visualisasi BQ 4 (Bar Chart) :** Dua chart horizontal membandingkan 10 kategori bergaji tertinggi vs terendah secara bersamaan.

In [55]:
# Visualisasi BQ 4 — Dual-panel bar chart

top10 = role_stats.head(10).copy()
bot10 = role_stats.tail(10).sort_values('median_gaji').copy()

def _format_rp(val):
    return f'Rp {val/1_000_000:.1f}Jt'

fig4a = make_subplots(
    rows=1, cols=2,
    subplot_titles=['10 Peran Gaji Tertinggi', '10 Peran Gaji Terendah'],
    shared_xaxes=False
)

fig4a.add_trace(
    go.Bar(
        x=top10['median_gaji'],
        y=top10[GLINTS_ROLE_COL],
        orientation='h',
        marker_color=COLOR_WFH,
        text=top10['median_gaji'].apply(_format_rp),
        textposition='inside',
        insidetextanchor='end',
        name='Tertinggi'
    ),
    row=1, col=1
)

fig4a.add_trace(
    go.Bar(
        x=bot10['median_gaji'],
        y=bot10[GLINTS_ROLE_COL],
        orientation='h',
        marker_color=COLOR_THRESHOLD,
        text=bot10['median_gaji'].apply(_format_rp),
        textposition='inside',
        insidetextanchor='end',
        name='Terendah'
    ),
    row=1, col=2
)

fig4a.update_layout(
    title_text='BQ 4 — Median Gaji per Kategori Peran (Top & Bottom 10)',
    template=TEMPLATE,
    height=560,
    showlegend=False
)
fig4a.update_xaxes(tickformat='.0s', dtick=5_000_000, row=1, col=1)
fig4a.update_xaxes(tickformat='.0s', dtick=5_000_000, row=1, col=2)
fig4a.show()

**Visualisasi BQ 4 (Box Plot):** Distribusi penuh gaji untuk 10 kategori teratas dan memperlihatkan sebaran, median, dan outlier setiap kategori.

In [56]:
# Parse gaji jika belum ada kolom salary_median
if 'salary_median' not in df_glints.columns:
    def _parse_gaji_bq4(series):
        clean = (
            series.astype(str)
            .str.replace(r'Rp', '', regex=False)
            .str.replace(r'/[Bb]ulan', '', regex=True)
            .str.replace(r'\.', '', regex=True)
            .str.strip()
        )
        split = clean.str.split(r'\s*-\s*', expand=True, regex=True)
        lo = pd.to_numeric(split[0].str.strip(), errors='coerce')
        hi = pd.to_numeric(split[1].str.strip(), errors='coerce') if split.shape[1] > 1 else lo
        return (lo + hi) / 2
    df_glints = df_glints.copy()
    df_glints['salary_median'] = _parse_gaji_bq4(df_glints['gaji'])

# Top 10 kategori peran berdasarkan median gaji (filter min 5 listing)
role_med = (
    df_glints[df_glints['salary_median'].notna()]
    .groupby(GLINTS_ROLE_COL)
    .agg(median_gaji=('salary_median', 'median'), count=(GLINTS_ROLE_COL, 'count'))
    .query('count >= 5')
    .sort_values('median_gaji', ascending=False)
    .head(10)
)
top10_roles_box = role_med.index.tolist()

df_box = df_glints[
    df_glints[GLINTS_ROLE_COL].isin(top10_roles_box) &
    df_glints['salary_median'].notna()
].copy()

# Filter outlier ekstrem (> 100 juta) agar boxplot terbaca
df_box = df_box[df_box['salary_median'] <= 100_000_000]

fig4b = px.box(
    df_box,
    x='salary_median',
    y=GLINTS_ROLE_COL,
    orientation='h',
    points='outliers',
    title='BQ 4 — Rentang Gaji (Box Plot) — Top 10 Kategori Peran',
    labels={'salary_median': 'Gaji Median (Rp)', GLINTS_ROLE_COL: 'Kategori Peran'},
    color=GLINTS_ROLE_COL,
    color_discrete_sequence=COLOR_SEQ,
    category_orders={GLINTS_ROLE_COL: top10_roles_box}
)
fig4b.update_layout(
    template=TEMPLATE,
    height=560,
    showlegend=False,
    xaxis=dict(
        tickformat='.2s',
        title='Gaji Median (Rp)'
    )
)
fig4b.show()


---
### BQ 5 Gap Pendidikan: Kandidat vs. Syarat Lowongan

**Pertanyaan:** Di level pendidikan mana selisih distribusi CV vs Job melebihi 15 poin persentase?

**Chart:** Grouped Bar Chart Horizontal dengan highlight gap > 15 poin

Membandingkan distribusi jenjang pendidikan kandidat (`Dataset_CV`) dengan persyaratan pendidikan minimum lowongan (`Dataset_Job`), kemudian menghitung selisih absolut.

In [57]:
cv_pct  = (df_cv[CV_EDUCATION_COL].value_counts(normalize=True) * 100).round(2)
job_pct = (df_job['pendidikan'].value_counts(normalize=True) * 100).round(2)

comparison = pd.concat([cv_pct, job_pct], axis=1, keys=['Kandidat CV (%)', 'Syarat Job (%)']).fillna(0)
comparison['Selisih (poin)'] = (comparison['Kandidat CV (%)'] - comparison['Syarat Job (%)']).abs().round(2)
comparison = comparison.sort_values('Kandidat CV (%)', ascending=False)

big_gap     = comparison[comparison['Selisih (poin)'] > 15]
max_gap_lvl = comparison['Selisih (poin)'].idxmax()
answer      = (
    f'Gap terbesar: {max_gap_lvl} ({comparison.loc[max_gap_lvl, "Selisih (poin)"]} poin) | '
    f'Ada gap >15 poin? {"YA" if not big_gap.empty else "TIDAK"}'
)

print_section('BQ 5 — Perbandingan Distribusi Pendidikan CV vs Job', comparison, answer)

------------------------------------------------------------
  BQ 5 — Perbandingan Distribusi Pendidikan CV vs Job
------------------------------------------------------------


,Kandidat CV (%),Syarat Job (%),Selisih (poin)
pendidikan,,,
Magister (S2),79.35,1.73,77.62
tidak ada pendidikan spesifik,8.86,0.00,8.86
Sarjana (S1),6.59,8.71,2.12
PhD (S3),3.77,0.00,3.77
Diploma 3 (D3),0.52,0.00,0.52
Diploma 1/2 (D1/D2),0.46,0.00,0.46
PhD (S3)),0.23,0.00,0.23
SMA/SMK,0.22,1.80,1.58
terbuka untuk semua jenjang dan jurusan,0.00,85.79,85.79



  JAWABAN : Gap terbesar: terbuka untuk semua jenjang dan jurusan (85.79 poin) | Ada gap >15 poin? YA



**Visualisasi BQ 5:** Grouped bar chart horizontal menampilkan distribusi kandidat (biru) vs syarat lowongan (oranye) berdampingan per jenjang pendidikan.

In [58]:
# Visualisasi BQ 5
df_plot5 = comparison.reset_index().rename(columns={'index': 'Pendidikan'})
df_plot5 = comparison.reset_index()
if df_plot5.columns[0] != 'Pendidikan':
    df_plot5 = df_plot5.rename(columns={df_plot5.columns[0]: 'Pendidikan'})

df_melt5 = df_plot5.melt(
    id_vars='Pendidikan',
    value_vars=['Kandidat CV (%)', 'Syarat Job (%)'],
    var_name='Sumber',
    value_name='Persen'
)

fig5 = px.bar(df_melt5, x='Persen', y='Pendidikan', color='Sumber',
              barmode='group', orientation='h',
              title='BQ 5 — Gap Distribusi Pendidikan CV vs Job')
fig5.show()

---
### BQ 6 - Top 15 Skill Paling Sering di Glints_Job

**Pertanyaan :** Apakah ada skill yang muncul di > 30% total lowongan?

**Chart :** Horizontal Bar Chart + garis vertikal threshold 30%

Kolom skill dipecah per-koma menggunakan `split_skills()`, kemudian dihitung frekuensi kemunculan relatif terhadap total jumlah lowongan.

In [59]:
total_jobs = len(df_glints)
all_skills = split_skills(df_glints[GLINTS_SKILL_COL])

skill_freq = all_skills.value_counts().reset_index()
skill_freq.columns = ['Skill', 'Jumlah']
skill_freq['Persen (%)'] = (skill_freq['Jumlah'] / total_jobs * 100).round(2)

dominant = skill_freq[skill_freq['Persen (%)'] >= 30]
top1     = skill_freq.iloc[0]

if not dominant.empty:
    answer = (
        f'Ada {len(dominant)} skill dominan (>30%). '
        f'Skill paling dominan: "{dominant.iloc[0]["Skill"]}" ({dominant.iloc[0]["Persen (%)"]}%)'
    )
else:
    answer = f'Tidak ada skill >30%. Top skill: "{top1["Skill"]}" ({top1["Persen (%)"]:.2f}%)'

print_section('BQ 6 — Top 15 Skill Glints_Job', skill_freq.head(15), answer)

------------------------------------------------------------
  BQ 6 — Top 15 Skill Glints_Job
------------------------------------------------------------


,Skill,Jumlah,Persen (%)
0,MySQL,88,15.58
1,JavaScript,84,14.87
2,Python,82,14.51
3,PostgreSQL,69,12.21
4,React.js,63,11.15
5,Microsoft SQL Server,63,11.15
6,IT Support,60,10.62
7,Node.js,58,10.27
8,SQL,57,10.09
9,Network Troubleshooting,55,9.73



  JAWABAN : Tidak ada skill >30%. Top skill: "MySQL" (15.58%)



**Visualisasi BQ 6:** Bar chart horizontal disertai garis merah putus-putus pada 30% sebagai batas skill dominan.

In [60]:
# Visualisasi BQ 6
top15 = skill_freq.head(15).sort_values('Persen (%)')
threshold = 30

fig6 = px.bar(top15, x='Persen (%)', y='Skill', orientation='h',
              title='BQ 6 — Top 15 Skill Glints_Job')
fig6.add_vline(x=threshold, line_dash='dash', line_color='red',
               annotation_text=f'{threshold}% threshold')
fig6.show()

---
### BQ 7 - Rata-rata Skill per Bucket Pengalaman

**Pertanyaan :** Apakah kandidat senior (5+ th) memiliki jumlah skill ≥ 2× dibanding entry-level (0–1 th)?

**Chart :** Bar Chart vertikal + Line overlay (tren)

Menghitung jumlah skill (`skill_count`) setiap kandidat dengan memecah kolom skill per-koma, lalu mengelompokkan berdasarkan bucket pengalaman dan menghitung rata-rata.

In [61]:
if 'skill_count' not in df_cv.columns:
    df_cv['skill_count'] = df_cv[CV_SKILL_COL].fillna('').apply(
        lambda x: len([s for s in str(x).split(',') if s.strip()])
    )

if 'experience_bucket' not in df_cv.columns:
    if CV_EXPERIENCE_COL in df_cv.columns:
        df_cv[CV_EXPERIENCE_COL] = pd.to_numeric(df_cv[CV_EXPERIENCE_COL], errors='coerce')
    else:
        src = 'pengalaman' if 'pengalaman' in df_cv.columns else None
        if src is None:
            df_cv[CV_EXPERIENCE_COL] = np.nan
        else:
            df_cv[CV_EXPERIENCE_COL] = (
                df_cv[src].astype(str).str.lower().str.extract(r'(\d+(?:\.\d+)?)')[0]
            )
            df_cv[CV_EXPERIENCE_COL] = pd.to_numeric(df_cv[CV_EXPERIENCE_COL], errors='coerce')
    df_cv['experience_bucket'] = pd.cut(
        df_cv[CV_EXPERIENCE_COL],
        bins=[-1, 1, 2, 5, 100],
        labels=['Entry (0-1 th)', 'Junior (1-2 th)', 'Mid (2-5 th)', 'Senior (5+ th)']
    )

agg = (
    df_cv.groupby('experience_bucket', observed=True)['skill_count']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
agg.columns = ['Bucket', 'Mean Skill', 'Std', 'Jumlah Kandidat']
agg = agg.round(2)

entry_mean  = float(agg.loc[agg['Bucket'].astype(str).str.contains('Entry'),  'Mean Skill'].values[0])
senior_mean = float(agg.loc[agg['Bucket'].astype(str).str.contains('Senior'), 'Mean Skill'].values[0])
ratio       = round(senior_mean / entry_mean, 2) if entry_mean > 0 else float('inf')

answer = (
    f'Senior >= 2x skill dari entry? {"YA" if ratio >= 2 else "TIDAK"} '
    f'(rasio: {ratio}x | Entry: {entry_mean}, Senior: {senior_mean})'
)

print_section('BQ 7 — Rata-rata Skill per Bucket Pengalaman', agg, answer)

------------------------------------------------------------
  BQ 7 — Rata-rata Skill per Bucket Pengalaman
------------------------------------------------------------


,Bucket,Mean Skill,Std,Jumlah Kandidat
0,Entry (0-1 th),14.15,11.33,98
1,Junior (1-2 th),17.73,12.27,144
2,Mid (2-5 th),21.57,10.82,7816
3,Senior (5+ th),21.78,14.07,2534



  JAWABAN : Senior >= 2x skill dari entry? TIDAK (rasio: 1.54x | Entry: 14.15, Senior: 21.78)



**Visualisasi BQ 7:** Bar chart menampilkan mean skill tiap bucket beserta error bar ±1 std; line overlay menunjukkan tren peningkatan skill seiring pengalaman.

In [62]:
# Visualisasi BQ 7 — Bar + Line overlay
fig7 = go.Figure()
fig7.add_trace(go.Bar(x=agg['Bucket'], y=agg['Mean Skill'], name='Mean Skill',
                      error_y=dict(type='data', array=agg['Std'].tolist())))
fig7.add_trace(go.Scatter(x=agg['Bucket'], y=agg['Mean Skill'], mode='lines+markers',
                           name='Tren', line=dict(dash='dot')))
fig7.update_layout(title='BQ 7 — Mean Skill per Bucket Pengalaman',
                   xaxis_title='Bucket', yaxis_title='Rata-rata Skill')
fig7.show()

---
### BQ 8 - Dominasi Tipe Waktu Kerja per Kategori Peran

**Pertanyaan :** Apakah tipe Penuh Waktu mendominasi > 70% di setiap kategori peran?

**Chart :** 100% Stacked Horizontal Bar Chart dengan garis 70%

Membuat pivot tabel komposisi tipe waktu kerja per kategori peran, lalu mengecek kategori mana yang proporsi Penuh Waktu-nya di bawah 70%.

In [63]:
# pilih kolom tipe waktu yang benar (fallback ke alternatif yang ada)
work_time_col = GLINTS_WORK_TIME_COL if GLINTS_WORK_TIME_COL in df_glints.columns else \
    next((c for c in df_glints.columns if 'tipe' in c.lower() and 'waktu' in c.lower()), None)
if work_time_col is None:
    work_time_col = next((c for c in df_glints.columns if 'waktu' in c.lower()), None)

if work_time_col:
    pivot_raw = df_glints.groupby([GLINTS_ROLE_COL, work_time_col]).size().unstack(fill_value=0)
else:
    pivot_raw = pd.DataFrame()
    print(f'Warning: kolom tipe waktu tidak ditemukan (dicari "{GLINTS_WORK_TIME_COL}" dan alternatif).')
pivot_pct = pivot_raw.div(pivot_raw.sum(axis=1), axis=0).mul(100).round(2)

penuh_col = next((c for c in pivot_pct.columns if 'penuh' in str(c).lower()), None)

if penuh_col:
    outliers = pivot_pct[pivot_pct[penuh_col] < 70].index.tolist()
    answer   = (
        f'Penuh Waktu >70% di semua kategori? {"YA" if not outliers else "TIDAK"} '
        f'({len(outliers)} kategori tidak memenuhi)'
    )
else:
    outliers = []
    answer   = 'Kolom Penuh Waktu tidak ditemukan. Cek nilai kolom tipe_waktu.'

print_section('BQ 8 — Tipe Waktu Kerja per Kategori Peran (%)', pivot_pct, answer)

------------------------------------------------------------
  BQ 8 — Tipe Waktu Kerja per Kategori Peran (%)
------------------------------------------------------------


tipe_waktu_kerja,Freelance,Kontrak,Magang,Paruh Waktu,Penuh Waktu
kategori_peran,,,,,
AI Engineer,2.78,19.44,5.56,0.00,72.22
Admin,0.00,0.00,0.00,0.00,100.00
Backend Developer,1.96,35.29,9.80,0.00,52.94
Business Analyst,0.00,50.00,0.00,0.00,50.00
Cybersecurity,10.00,20.00,0.00,0.00,70.00
Data Analyst,0.00,18.75,18.75,0.00,62.50
Data Architect,0.00,100.00,0.00,0.00,0.00
Data Engineer,0.00,21.43,0.00,0.00,78.57
Data Scientist,0.00,16.67,0.00,0.00,83.33



  JAWABAN : Penuh Waktu >70% di semua kategori? TIDAK (19 kategori tidak memenuhi)



**Visualisasi BQ 8:** 100% stacked bar horizontal dengan garis merah pada 70% dan memudahkan identifikasi kategori yang memiliki proporsi Penuh Waktu rendah.

In [64]:
# Visualisasi BQ 8 — 100% Stacked Bar
# gunakan kolom yang terdeteksi (work_time_col); rename ke GLINTS_WORK_TIME_COL agar konsisten di plotting
if work_time_col is None or work_time_col not in df_glints.columns:
    df_plot8 = pd.DataFrame(columns=[GLINTS_ROLE_COL, GLINTS_WORK_TIME_COL, 'Jumlah'])
else:
    df_plot8 = (
        df_glints.groupby([GLINTS_ROLE_COL, work_time_col])
        .size().reset_index(name='Jumlah')
    )
    if work_time_col != GLINTS_WORK_TIME_COL:
        df_plot8 = df_plot8.rename(columns={work_time_col: GLINTS_WORK_TIME_COL})
totals8  = df_plot8.groupby(GLINTS_ROLE_COL)['Jumlah'].transform('sum')
df_plot8['Persen (%)'] = (df_plot8['Jumlah'] / totals8 * 100).round(2)

fig8 = px.bar(df_plot8, x='Persen (%)', y=GLINTS_ROLE_COL, color=GLINTS_WORK_TIME_COL,
              orientation='h', barmode='stack',
              title='BQ 8 — Tipe Waktu Kerja per Kategori Peran (100% Stacked)')
fig8.add_vline(x=70, line_dash='dash', line_color='red', annotation_text='70%')
fig8.show()

---
### BQ 9 - Lowongan dengan Batasan Gender/Usia

**Pertanyaan :** Berapa persen lowongan yang mencantumkan syarat gender/usia spesifik? Apakah melebihi 20%?

**Chart :** Pie Chart + Horizontal Bar Chart (top posisi)

Sebuah lowongan dianggap memiliki batasan jika kolom gender berisi nilai selain 'Semua'/'Tidak Ditentukan' ATAU kolom usia minimum tidak kosong.

In [65]:
# Kolom di Dataset_Job: 'gender' dan 'usia'
# Nilai "tanpa ketentuan" / "tanpa batasan usia" = TIDAK ada batasan

GENDER_NO_RESTRICTION = {'tanpa ketentuan', 'nan', '', 'tidak ada'}
AGE_NO_RESTRICTION    = {'tanpa batasan usia', 'nan', '', 'tidak ada'}

has_gender_restriction = (
    df_job['gender'].notna() &
    ~df_job['gender'].astype(str).str.strip().str.lower().isin(GENDER_NO_RESTRICTION)
)

has_age_restriction = (
    df_job['usia'].notna() &
    ~df_job['usia'].astype(str).str.strip().str.lower().isin(AGE_NO_RESTRICTION)
)

df_job9 = df_job.copy()
df_job9['has_restriction'] = has_gender_restriction | has_age_restriction

total       = len(df_job9)
restricted  = int(df_job9['has_restriction'].sum())
unrestricted= total - restricted
pct_restricted = round(restricted / total * 100, 2)

print_section(
    'BQ 9 — Lowongan dengan Batasan Gender/Usia',
    pd.DataFrame({
        'Kategori': ['Ada Batasan', 'Tanpa Batasan', 'Total'],
        'Jumlah'  : [restricted, unrestricted, total],
        'Persen (%)': [pct_restricted, round(100 - pct_restricted, 2), 100.0]
    }),
    answer=f'{pct_restricted:.2f}% lowongan memiliki batasan gender/usia | Melebihi 20%? {"YA" if pct_restricted > 20 else "TIDAK"}'
)


------------------------------------------------------------
  BQ 9 — Lowongan dengan Batasan Gender/Usia
------------------------------------------------------------


,Kategori,Jumlah,Persen (%)
0,Ada Batasan,227,2.06
1,Tanpa Batasan,10800,97.94
2,Total,11027,100.00



  JAWABAN : 2.06% lowongan memiliki batasan gender/usia | Melebihi 20%? TIDAK



**Visualisasi BQ 9:** Pie chart menampilkan proporsi lowongan dengan/tanpa batasan, bar chart horizontal menampilkan posisi yang paling banyak mencantumkan batasan.

In [66]:
# Pie Chart — proporsi ada/tidak batasan
fig9a = go.Figure(go.Pie(
    labels=['Ada Batasan Gender/Usia', 'Tidak Ada Batasan'],
    values=[restricted, unrestricted],
    hole=0.4,
    marker=dict(colors=['#E53935', '#4CAF50']),
    textinfo='label+percent+value'
))
fig9a.update_layout(
    title_text='BQ 9 — Proporsi Lowongan dengan Batasan Gender/Usia',
    template=TEMPLATE,
    height=420
)
fig9a.show()

# Bar Chart — Top 15 posisi yang membatasi
top_positions = (
    df_job9[df_job9['has_restriction']]['posisi']
    .value_counts()
    .head(15)
    .reset_index()
)
top_positions.columns = ['Posisi', 'Jumlah Lowongan']
top_positions = top_positions.sort_values('Jumlah Lowongan', ascending=True)

fig9b = px.bar(
    top_positions,
    x='Jumlah Lowongan',
    y='Posisi',
    orientation='h',
    text='Jumlah Lowongan',
    title='BQ 9 — Top 15 Posisi dengan Batasan Gender/Usia',
    color='Jumlah Lowongan',
    color_continuous_scale='Reds',
    category_orders={'Posisi': top_positions['Posisi'].tolist()}
)
fig9b.update_traces(textposition='outside')
fig9b.update_layout(template=TEMPLATE, height=500, showlegend=False)
fig9b.show()


---
### BQ 10 Perbandingan Median Gaji WFO/WFH/Hybrid per Kategori

**Pertanyaan:** Dari 10 kategori peran terbanyak, bagaimana perbandingan median gaji antar sistem kerja?

**Chart:** Grouped Bar Chart Horizontal + Heatmap

Memfilter 10 kategori peran dengan jumlah lowongan terbanyak, lalu menghitung median gaji per kombinasi kategori × sistem kerja (WFO/WFH/Hybrid).

In [67]:
if 'salary_median' not in df_glints.columns:
    df_glints['salary_median'] = df_glints[[GLINTS_SALARY_MIN, GLINTS_SALARY_MAX]].mean(axis=1)

top10_roles = df_glints[GLINTS_ROLE_COL].value_counts().head(10).index.tolist()
df_filtered = df_glints[
    df_glints[GLINTS_ROLE_COL].isin(top10_roles) &
    df_glints['salary_median'].notna()
]

pivot_gaji = (
    df_filtered.groupby([GLINTS_ROLE_COL, GLINTS_WORK_SYS_COL])['salary_median']
    .median().unstack(fill_value=0) / 1_000_000
).round(2)
pivot_gaji.columns.name = 'Sistem Kerja'
pivot_gaji.index.name   = 'Kategori Peran'

overall = (
    df_filtered.groupby(GLINTS_WORK_SYS_COL)['salary_median']
    .median().sort_values(ascending=False)
)
overall_tbl = pd.DataFrame({
    'Sistem Kerja'       : overall.index,
    'Median Gaji (Juta)' : (overall.values / 1e6).round(2)
})

answer = f'Sistem kerja gaji tertinggi: "{overall.index[0]}" (Rp {overall.iloc[0]/1e6:.2f} Jt)'

print_section('BQ 10 — Median Gaji (Juta Rp) per Sistem Kerja & Kategori (Top 10)', pivot_gaji, answer=None)
print_section('BQ 10 — Rata-rata Median Gaji per Sistem Kerja', overall_tbl, answer)

------------------------------------------------------------
  BQ 10 — Median Gaji (Juta Rp) per Sistem Kerja & Kategori (Top 10)
------------------------------------------------------------


Sistem Kerja,Hybrid,Kerja di kantor,Remote/Dari rumah
Kategori Peran,,,
AI Engineer,2.50,2.50,2.00
Backend Developer,5.50,3.00,2.50
Data Analyst,0.50,2.50,0.00
Data Engineer,5.00,3.75,0.00
DevOps Engineer,3.50,3.00,5.00
Frontend Developer,4.75,2.50,3.00
Full Stack Developer,1.50,2.00,2.50
IT Support,6.50,1.75,0.75
Network Engineer,2.75,2.50,0.00



------------------------------------------------------------
  BQ 10 — Rata-rata Median Gaji per Sistem Kerja
------------------------------------------------------------


,Sistem Kerja,Median Gaji (Juta)
0,Hybrid,3.25
1,Kerja di kantor,2.50
2,Remote/Dari rumah,2.50



  JAWABAN : Sistem kerja gaji tertinggi: "Hybrid" (Rp 3.25 Jt)



**Visualisasi BQ 10 (Grouped Bar):** Membandingkan median gaji WFO/WFH/Hybrid untuk 10 kategori peran terbesar secara bersamaan.

In [68]:
# Visualisasi BQ 10 — Grouped Bar
df_plot10 = (
    df_filtered.groupby([GLINTS_ROLE_COL, GLINTS_WORK_SYS_COL])['salary_median']
    .median().reset_index()
)
df_plot10['salary_median'] = (df_plot10['salary_median'] / 1e6).round(2)
df_plot10.columns = ['Kategori Peran', 'Sistem Kerja', 'Median Gaji (Juta)']

fig10a = px.bar(df_plot10, x='Median Gaji (Juta)', y='Kategori Peran',
                color='Sistem Kerja', barmode='group', orientation='h',
                title='BQ 10 — Median Gaji per Sistem Kerja (Top 10 Kategori)')
fig10a.show()

**Visualisasi BQ 10 (Heatmap) :** Heatmap memberikan gambaran menyeluruh pola gaji lintas kategori dan sistem kerja — sel lebih terang = gaji lebih tinggi.

In [69]:
# Visualisasi BQ 10 — Heatmap
fig10b = px.imshow(pivot_gaji,
                   labels=dict(x='Sistem Kerja', y='Kategori Peran', color='Median Gaji (Juta Rp)'),
                   title='BQ 10 — Heatmap Median Gaji per Sistem Kerja & Kategori',
                   text_auto=True, aspect='auto')
fig10b.show()

---
## 4. Ringkasan Jawaban Semua Business Questions

Menggabungkan seluruh jawaban dari BQ 1–10 ke dalam satu ringkasan terkonsolidasi untuk kemudahan pelaporan dan presentasi.

In [70]:
sep = '=' * 70
print(sep)
print('  RINGKASAN BUSINESS QUESTIONS — RECRUITMENT ANALYTICS')
print('  DBS Foundation Data Science Capstone 2026')
print(sep)

answers = [
    ('BQ 1',  f'S1 mendominasi >50%? {s1_pct:.2f}%'),
    ('BQ 2',  f'Entry-level >40%? {entry_pct:.2f}%'),
    ('BQ 3',  f'Top 5 kota: {" > ".join(top5)}'),
    ('BQ 4',  f'Gaji tertinggi: {role_stats.iloc[0][GLINTS_ROLE_COL]} (Rp {top_med/1e6:.2f} Jt) | Terendah: {role_stats.iloc[-1][GLINTS_ROLE_COL]} (Rp {bot_med/1e6:.2f} Jt)'),
    ('BQ 5',  f'Gap terbesar: {max_gap_lvl} ({comparison.loc[max_gap_lvl, "Selisih (poin)"]} poin) | Gap >15 poin? {"YA" if not big_gap.empty else "TIDAK"}'),
    ('BQ 6',  f'Skill dominan >30%? {"YA" if not dominant.empty else "TIDAK"} | Top skill: {top1["Skill"]} ({top1["Persen (%)"]:.2f}%)'),
    ('BQ 7',  f'Senior >= 2x skill dari entry? {"YA" if ratio >= 2 else "TIDAK"} (rasio: {ratio}x)'),
    ('BQ 8',  f'Penuh Waktu >70% semua kategori? {"YA" if not outliers else "TIDAK"} ({len(outliers)} outlier)'),
    ('BQ 9',  f'Batasan >20%? {"YA" if pct > 20 else "TIDAK"} ({pct:.2f}%) | Top posisi: {top_pos.iloc[0]["Posisi"]}'),
    ('BQ 10', f'Sistem kerja gaji tertinggi: {overall.index[0]} (Rp {overall.iloc[0]/1e6:.2f} Jt)'),
]

for bq, ans in answers:
    print(f'\n  [{bq}] {ans}')

print(f'\n{sep}')

  RINGKASAN BUSINESS QUESTIONS — RECRUITMENT ANALYTICS
  DBS Foundation Data Science Capstone 2026


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().